In [0]:
# DBTITLE 1,Parameters
CATALOG = "workspace"             # <-- ajustar
SCHEMA  = "default"              # <-- ajustar
TABLE   = "customer_profiles"

TEST_SIZE    = 0.25
RANDOM_STATE = 42

FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"
print(f"Tabla origen: {FULL_TABLE}")

In [0]:
# DBTITLE 1,Cargar dataset de modelado desde Unity Catalog
import pandas as pd
import numpy as np

sdf = spark.table(FULL_TABLE)
print(f"Filas: {sdf.count():,}  Columnas: {len(sdf.columns)}")
df = sdf.toPandas()
df.head()

In [0]:
# DBTITLE 1,Preparar matriz de features y target
DROP_COLS = ['customer', 'rfm_code', 'top_brand', 'rfm_segment', 'brand_affinity']

# Encoding simple para la marca principal
df['brand_code'] = df['top_brand'].astype('category').cat.codes

# One-hot encoding para rfm_segment
df = pd.concat([df, pd.get_dummies(df['rfm_segment'], prefix='seg')], axis=1)

# One-hot encoding para brand_affinity
df = pd.concat([df, pd.get_dummies(df['brand_affinity'], prefix='affinity')], axis=1)

X = (df.drop(columns=[c for c in DROP_COLS if c in df.columns])
       .select_dtypes(include=[np.number, 'bool']))

print(f"X shape: {X.shape}")
print(f"# features: {X.shape[1]}")
print(f"\nNote: This dataset does not have a 'target' column.")
print(f"Please define what you want to predict (e.g., churn, segment, purchase likelihood).")